In [1]:
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# Ensure working directory is project root
current_path = Path.cwd()
if current_path.name == 'notebook':
    os.chdir(current_path.parent)

data_dir = Path('data/raw')
if not data_dir.exists():
    data_dir = Path('dataset/raw')

# print(f"Active working directory: {Path.cwd()}")
# print(f"Data directory: {data_dir.resolve()} (Exists: {data_dir.exists()})")

# Load column definitions from features CSV
features_file = data_dir / 'NUSW-NB15_features.csv'
features_df = pd.read_csv(features_file, encoding='latin1')

# Strip whitespace from column names to avoid formatting issues (e.g., 'ct_src_ ltm')
features = [str(name).strip() for name in features_df['Name']]

In [3]:
# Define explicit dtypes for columns known to contain mixed/string values
# to avoid pandas DtypeWarning and optimize memory usage
dtype_spec = {
    'sport': str,
    'dsport': str,
    'ct_ftp_cmd': str,
    'attack_cat': str,
    'Label': np.int64
}

print("Loading UNSW-NB15 dataset splits (df1 - df4)...")
df1 = pd.read_csv(data_dir / 'UNSW-NB15_1.csv', names=features, dtype=dtype_spec, low_memory=False)
df2 = pd.read_csv(data_dir / 'UNSW-NB15_2.csv', names=features, dtype=dtype_spec, low_memory=False)
df3 = pd.read_csv(data_dir / 'UNSW-NB15_3.csv', names=features, dtype=dtype_spec, low_memory=False)
df4 = pd.read_csv(data_dir / 'UNSW-NB15_4.csv', names=features, dtype=dtype_spec, low_memory=False)

dfs = {'df1': df1, 'df2': df2, 'df3': df3, 'df4': df4}
print("All 4 dataframes loaded successfully.")

Loading UNSW-NB15 dataset splits (df1 - df4)...
All 4 dataframes loaded successfully.


In [4]:
df4.info()

<class 'pandas.DataFrame'>
RangeIndex: 440044 entries, 0 to 440043
Data columns (total 49 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   srcip             440044 non-null  str    
 1   sport             440044 non-null  str    
 2   dstip             440044 non-null  str    
 3   dsport            440044 non-null  str    
 4   proto             440044 non-null  str    
 5   state             440044 non-null  str    
 6   dur               440044 non-null  float64
 7   sbytes            440044 non-null  int64  
 8   dbytes            440044 non-null  int64  
 9   sttl              440044 non-null  int64  
 10  dttl              440044 non-null  int64  
 11  sloss             440044 non-null  int64  
 12  dloss             440044 non-null  int64  
 13  service           440044 non-null  str    
 14  Sload             440044 non-null  float64
 15  Dload             440044 non-null  float64
 16  Spkts             440044 non-nu

In [5]:
# hexadecimal port number
df1[df1['dsport']=='0x20205321']

,srcip,sport,dstip,dsport,proto,state,dur,sbytes,dbytes,sttl,...,ct_ftp_cmd,ct_srv_src,ct_srv_dst,ct_dst_ltm,ct_src_ ltm,ct_src_dport_ltm,ct_dst_sport_ltm,ct_dst_src_ltm,attack_cat,Label
82531,175.45.176.1,-,149.171.126.12,0x20205321,esp,INT,0.000000,200,0,254,...,0,12,12,4,4,2,1,4,NaN,0
82532,175.45.176.1,0,149.171.126.12,0x20205321,esp,INT,0.000004,200,0,254,...,0,12,12,4,4,2,3,4,NaN,0


In [6]:
df_filt = df1.dropna()
df_filt

,srcip,sport,dstip,dsport,proto,state,dur,sbytes,dbytes,sttl,...,ct_ftp_cmd,ct_srv_src,ct_srv_dst,ct_dst_ltm,ct_src_ ltm,ct_src_dport_ltm,ct_dst_sport_ltm,ct_dst_src_ltm,attack_cat,Label
20,175.45.176.3,21223,149.171.126.18,32780,udp,INT,0.000021,728,0,254,...,0,1,1,1,1,1,1,1,Exploits,1
21,175.45.176.2,23357,149.171.126.16,80,tcp,FIN,0.240139,918,25552,62,...,0,3,2,2,1,1,1,1,Exploits,1
22,175.45.176.0,13284,149.171.126.16,80,tcp,FIN,2.390390,1362,268,254,...,0,5,2,2,1,1,1,1,Reconnaissance,1
39,175.45.176.2,13792,149.171.126.16,5555,tcp,FIN,0.175190,8168,268,254,...,0,1,1,1,1,1,1,1,Exploits,1
40,175.45.176.2,26939,149.171.126.10,80,tcp,FIN,0.190600,844,268,254,...,0,3,1,1,1,1,1,1,Exploits,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
186498,175.45.176.1,58463,149.171.126.15,179,tcp,FIN,31.461020,924,708,254,...,0,6,6,2,2,2,1,2,Fuzzers,1
186499,175.45.176.1,58460,149.171.126.15,179,tcp,FIN,57.995934,1362,536,254,...,0,6,6,2,2,2,1,2,Fuzzers,1
186562,175.45.176.1,58967,149.171.126.15,179,tcp,FIN,32.361782,1128,622,254,...,0,6,6,2,2,2,1,2,Fuzzers,1
186658,175.45.176.1,58485,149.171.126.15,179,tcp,FIN,0.476639,470,354,254,...,0,8,8,2,2,2,1,2,Fuzzers,1


In [7]:
# check ipv6
import ipaddress
import pandas as pd

df = df_filt.copy()

def is_ipv6(val):
    try:
        # Handles strings like "2001:db8::1" or CIDR blocks like "2001:db8::/32"
        ip = ipaddress.ip_network(str(val).strip(), strict=False)
        return ip.version == 6
    except (ValueError, AttributeError):
        return False

# Check across a specific column
ipv6_mask = df["srcip"].apply(is_ipv6)
ipv6_rows = df[ipv6_mask]

print(f"Total IPv6 entries found: {ipv6_mask.sum()}")
print(ipv6_rows[["srcip", "dstip"]].head())

Total IPv6 entries found: 0
Empty DataFrame
Columns: [srcip, dstip]
Index: []


# Preprocessing

## Preprocess zeek features

feature comparison

In [8]:
# UNSW NB15 dataset
unsw_col = features_df['Name']
print(unsw_col)

0                srcip
1                sport
2                dstip
3               dsport
4                proto
5                state
6                  dur
7               sbytes
8               dbytes
9                 sttl
10                dttl
11               sloss
12               dloss
13             service
14               Sload
15               Dload
16               Spkts
17               Dpkts
18                swin
19                dwin
20               stcpb
21               dtcpb
22             smeansz
23             dmeansz
24         trans_depth
25         res_bdy_len
26                Sjit
27                Djit
28               Stime
29               Ltime
30             Sintpkt
31             Dintpkt
32              tcprtt
33              synack
34              ackdat
35     is_sm_ips_ports
36        ct_state_ttl
37    ct_flw_http_mthd
38        is_ftp_login
39          ct_ftp_cmd
40          ct_srv_src
41          ct_srv_dst
42          ct_dst_ltm
43         

In [9]:
# zeek logs
import pandas as pd
def load_zeek_conn_log(filepath):
    field_names = []
    with open(filepath, 'r') as f:
        for line in f:
            if line.startswith('#fields'):
                field_names = line.strip().split('\t')[1:]
                break
    df = pd.read_csv(
        filepath,
        sep='\t',
        comment='#',
        names=field_names,
        na_values='-',
        low_memory=False
    )
    return df

df_conn = load_zeek_conn_log('zeek_logs/example/conn.log')
print(f'loaded {len(df_conn)} connection records.')
zeek_col = df_conn.columns
print(zeek_col)

loaded 35 connection records.
Index(['ts', 'uid', 'id.orig_h', 'id.orig_p', 'id.resp_h', 'id.resp_p',
       'proto', 'service', 'duration', 'orig_bytes', 'resp_bytes',
       'conn_state', 'local_orig', 'local_resp', 'missed_bytes', 'history',
       'orig_pkts', 'orig_ip_bytes', 'resp_pkts', 'resp_ip_bytes',
       'tunnel_parents', 'ip_proto'],
      dtype='str')


feature mapping & engineering

In [10]:
excluded_col = ['sttl', 'dttl', 'sloss', 'dloss', 'swin', 'dwin', 'stcpb', 'dtcpb',
            'trans_depth', 'res_bdy_len', 'Sjit', 'Djit', 'Sintpkt', 'Dintpkt',
            'tcprtt', 'synack', 'ackdat', 'ct_flw_http_mthd', 'is_ftp_login', 'ct_ftp_cmd',
            'attack_cat']

def map_features(zeek_path):
    df_zeek = load_zeek_conn_log(zeek_path)
    df = pd.DataFrame()

    duration = df_zeek["duration"].fillna(0.0).astype(float).values
    orig_bytes = df_zeek["orig_bytes"].fillna(0).astype(float).values
    resp_bytes = df_zeek["resp_bytes"].fillna(0).astype(float).values
    orig_pkts = df_zeek["orig_pkts"].fillna(0).astype(float).values
    resp_pkts = df_zeek["resp_pkts"].fillna(0).astype(float).values
    ts = df_zeek["ts"].fillna(0.0).astype(float).values

    # Direct mappings
    df["sport"] = df_zeek["id.orig_p"]
    df["dsport"] = df_zeek["id.resp_p"]
    df["proto"] = df_zeek["proto"].astype(str)
    df["state"] = df_zeek["conn_state"].astype(str)
    df["dur"] = duration
    df["sbytes"] = orig_bytes
    df["dbytes"] = resp_bytes
    df["Spkts"] = orig_pkts
    df["Dpkts"] = resp_pkts
    df["service"] = df_zeek["service"].fillna("-").astype(str)
    df["Stime"] = ts

    # Derivable features
    df["Sload"] = np.divide(orig_bytes * 8, duration, out=np.zeros_like(orig_bytes, dtype=float), where=duration > 0)
    df["Dload"] = np.divide(resp_bytes * 8, duration, out=np.zeros_like(resp_bytes, dtype=float), where=duration > 0)
    df["smeansz"] = np.divide(orig_bytes, orig_pkts, out=np.zeros_like(orig_bytes, dtype=float), where=orig_pkts > 0)
    df["dmeansz"] = np.divide(resp_bytes, resp_pkts, out=np.zeros_like(resp_bytes, dtype=float), where=resp_pkts > 0)
    df["Ltime"] = ts + duration
    df["is_sm_ips_ports"] = (
        (df_zeek["id.orig_h"] == df_zeek["id.resp_h"]) &
        (df_zeek["id.orig_p"] == df_zeek["id.resp_p"])
    ).astype(int)

    # Windowed count aggregations (100-connection rolling window)
    n = len(df_zeek)
    ct_srv_src = np.zeros(n, dtype=int)
    ct_srv_dst = np.zeros(n, dtype=int)
    ct_dst_ltm = np.zeros(n, dtype=int)
    ct_src_ltm = np.zeros(n, dtype=int)
    ct_src_dport_ltm = np.zeros(n, dtype=int)
    ct_dst_sport_ltm = np.zeros(n, dtype=int)
    ct_dst_src_ltm = np.zeros(n, dtype=int)
    ct_state_ttl = np.zeros(n, dtype=int)

    orig_h = df_zeek["id.orig_h"].values
    resp_h = df_zeek["id.resp_h"].values
    orig_p = df_zeek["id.orig_p"].values
    resp_p = df_zeek["id.resp_p"].values
    services = df["service"].values
    states = df["state"].values

    for i in range(n):
        start_idx = max(0, i - 100)
        w_orig_h = orig_h[start_idx : i + 1]
        w_resp_h = resp_h[start_idx : i + 1]
        w_orig_p = orig_p[start_idx : i + 1]
        w_resp_p = resp_p[start_idx : i + 1]
        w_serv = services[start_idx : i + 1]
        w_state = states[start_idx : i + 1]

        ct_srv_src[i] = np.sum((w_serv == services[i]) & (w_orig_h == orig_h[i]))
        ct_srv_dst[i] = np.sum((w_serv == services[i]) & (w_resp_h == resp_h[i]))
        ct_dst_ltm[i] = np.sum(w_resp_h == resp_h[i])
        ct_src_ltm[i] = np.sum(w_orig_h == orig_h[i])
        ct_src_dport_ltm[i] = np.sum((w_orig_h == orig_h[i]) & (w_resp_p == resp_p[i]))
        ct_dst_sport_ltm[i] = np.sum((w_resp_h == resp_h[i]) & (w_orig_p == orig_p[i]))
        ct_dst_src_ltm[i] = np.sum((w_orig_h == orig_h[i]) & (w_resp_h == resp_h[i]))
        ct_state_ttl[i] = np.sum(w_state == states[i])

    df["ct_srv_src"] = ct_srv_src
    df["ct_srv_dst"] = ct_srv_dst
    df["ct_dst_ltm"] = ct_dst_ltm
    df["ct_src_ltm"] = ct_src_ltm
    df["ct_src_dport_ltm"] = ct_src_dport_ltm
    df["ct_dst_sport_ltm"] = ct_dst_sport_ltm
    df["ct_dst_src_ltm"] = ct_dst_src_ltm
    df["ct_state_ttl"] = ct_state_ttl

    df["sport_is_well_known"] = (df["sport"] < 1024).astype(int)
    df["dsport_is_well_known"] = (df["dsport"] < 1024).astype(int)
    df["sport_is_ephemeral"] = (df["sport"] >= 49152).astype(int)
    df["dsport_is_ephemeral"] = (df["dsport"] >= 49152).astype(int)

    critical_ports = {80, 443, 22, 23, 53, 445, 3389}
    df["dsport_is_critical"] = df["dsport"].isin(critical_ports).astype(int)

    df['label'] = 0

    return df

df_mapped = map_features('zeek_logs/example/conn.log')
print('Mapped dataframe shape:', df_mapped.shape)
df_mapped.head()

Mapped dataframe shape: (35, 31)


,sport,dsport,proto,state,dur,sbytes,dbytes,Spkts,Dpkts,service,...,ct_src_dport_ltm,ct_dst_sport_ltm,ct_dst_src_ltm,ct_state_ttl,sport_is_well_known,dsport_is_well_known,sport_is_ephemeral,dsport_is_ephemeral,dsport_is_critical,label
0,47742,443,tcp,SF,0.181797,98.0,0.0,5.0,4.0,-,...,1,1,1,1,0,1,0,0,1,0
1,5353,5353,udp,S0,0.000000,0.0,0.0,1.0,0.0,dns,...,1,1,1,1,0,0,0,0,0,0
2,57566,53,udp,SF,0.036839,58.0,123.0,1.0,1.0,dns,...,1,1,1,2,0,1,1,0,1,0
3,39662,53,udp,SF,0.036499,58.0,74.0,1.0,1.0,dns,...,2,1,2,3,0,1,0,0,1,0
4,47522,443,tcp,SF,0.348126,2095.0,3808.0,10.0,5.0,ssl,...,2,1,1,4,0,1,0,0,1,0


In [11]:
df_mapped.info()

<class 'pandas.DataFrame'>
RangeIndex: 35 entries, 0 to 34
Data columns (total 31 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   sport                 35 non-null     int64  
 1   dsport                35 non-null     int64  
 2   proto                 35 non-null     str    
 3   state                 35 non-null     str    
 4   dur                   35 non-null     float64
 5   sbytes                35 non-null     float64
 6   dbytes                35 non-null     float64
 7   Spkts                 35 non-null     float64
 8   Dpkts                 35 non-null     float64
 9   service               35 non-null     str    
 10  Stime                 35 non-null     float64
 11  Sload                 35 non-null     float64
 12  Dload                 35 non-null     float64
 13  smeansz               35 non-null     float64
 14  dmeansz               35 non-null     float64
 15  Ltime                 35 non-null   

## Preprocess UNSW-NB15 Dataset

In [12]:
df_combined = pd.concat([df1, df2, df3, df4])
df_combined.info()

<class 'pandas.DataFrame'>
Index: 2540047 entries, 0 to 440043
Data columns (total 49 columns):
 #   Column            Dtype  
---  ------            -----  
 0   srcip             str    
 1   sport             str    
 2   dstip             str    
 3   dsport            str    
 4   proto             str    
 5   state             str    
 6   dur               float64
 7   sbytes            int64  
 8   dbytes            int64  
 9   sttl              int64  
 10  dttl              int64  
 11  sloss             int64  
 12  dloss             int64  
 13  service           str    
 14  Sload             float64
 15  Dload             float64
 16  Spkts             int64  
 17  Dpkts             int64  
 18  swin              int64  
 19  dwin              int64  
 20  stcpb             int64  
 21  dtcpb             int64  
 22  smeansz           int64  
 23  dmeansz           int64  
 24  trans_depth       int64  
 25  res_bdy_len       int64  
 26  Sjit              float64
 27  

In [13]:
excluded_col = [
    "sttl", "dttl", "sloss", "dloss", "swin", "dwin", "stcpb", "dtcpb",
    "trans_depth", "res_bdy_len", "Sjit", "Djit", "Sintpkt", "Dintpkt",
    "tcprtt", "synack", "ackdat", "ct_flw_http_mthd", "is_ftp_login", "ct_ftp_cmd",
    "attack_cat", "Stime", "Ltime"
]

df_cleaned = df_combined.drop(columns=excluded_col) #drop columnns not in zeek logs
df_cleaned.info()

<class 'pandas.DataFrame'>
Index: 2540047 entries, 0 to 440043
Data columns (total 26 columns):
 #   Column            Dtype  
---  ------            -----  
 0   srcip             str    
 1   sport             str    
 2   dstip             str    
 3   dsport            str    
 4   proto             str    
 5   state             str    
 6   dur               float64
 7   sbytes            int64  
 8   dbytes            int64  
 9   service           str    
 10  Sload             float64
 11  Dload             float64
 12  Spkts             int64  
 13  Dpkts             int64  
 14  smeansz           int64  
 15  dmeansz           int64  
 16  is_sm_ips_ports   int64  
 17  ct_state_ttl      int64  
 18  ct_srv_src        int64  
 19  ct_srv_dst        int64  
 20  ct_dst_ltm        int64  
 21  ct_src_ ltm       int64  
 22  ct_src_dport_ltm  int64  
 23  ct_dst_sport_ltm  int64  
 24  ct_dst_src_ltm    int64  
 25  Label             int64  
dtypes: float64(3), int64(16), str(7

In [14]:
# check hexadecimal port count
sport_hex_count = df_combined['sport'].str.contains('0x').sum()
dsport_hex_count = df_combined['dsport'].str.contains('0x').sum()

print(f'sport: {sport_hex_count}, dsport: {dsport_hex_count}')

sport: 6, dsport: 297


In [15]:
# process hexadecimal port
def parse_port(val):
    if pd.isna(val) or str(val).strip() in ('-', '', 'NaN'):
        return 0
    val_str = str(val).strip()
    try:
        if val_str.startswith(('0x', '0X')):
            return int(val_str, 16)
        return int(float(val_str))
    except ValueError:
        return 0

df_cleaned['sport'] = df_cleaned['sport'].apply(parse_port)
df_cleaned['dsport'] = df_cleaned['dsport'].apply(parse_port)

In [16]:
df_cleaned = df_cleaned.drop(columns=["srcip", "dstip"], errors="ignore")
df_cleaned["sport"] = df_cleaned["sport"].apply(parse_port)
df_cleaned["dsport"] = df_cleaned["dsport"].apply(parse_port)

# feature engineering port numbers
df_cleaned["sport_is_well_known"] = (df_cleaned["sport"] < 1024).astype(int)
df_cleaned["dsport_is_well_known"] = (df_cleaned["dsport"] < 1024).astype(int)
df_cleaned["sport_is_ephemeral"] = (df_cleaned["sport"] >= 49152).astype(int)
df_cleaned["dsport_is_ephemeral"] = (df_cleaned["dsport"] >= 49152).astype(int)

critical_ports = {80, 443, 22, 23, 53, 445, 3389}
df_cleaned["dsport_is_critical"] = df_cleaned["dsport"].isin(critical_ports).astype(int)

In [17]:
df_cleaned.info()

<class 'pandas.DataFrame'>
Index: 2540047 entries, 0 to 440043
Data columns (total 29 columns):
 #   Column                Dtype  
---  ------                -----  
 0   sport                 int64  
 1   dsport                int64  
 2   proto                 str    
 3   state                 str    
 4   dur                   float64
 5   sbytes                int64  
 6   dbytes                int64  
 7   service               str    
 8   Sload                 float64
 9   Dload                 float64
 10  Spkts                 int64  
 11  Dpkts                 int64  
 12  smeansz               int64  
 13  dmeansz               int64  
 14  is_sm_ips_ports       int64  
 15  ct_state_ttl          int64  
 16  ct_srv_src            int64  
 17  ct_srv_dst            int64  
 18  ct_dst_ltm            int64  
 19  ct_src_ ltm           int64  
 20  ct_src_dport_ltm      int64  
 21  ct_dst_sport_ltm      int64  
 22  ct_dst_src_ltm        int64  
 23  Label                 in

In [18]:
df_mapped.info()

<class 'pandas.DataFrame'>
RangeIndex: 35 entries, 0 to 34
Data columns (total 31 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   sport                 35 non-null     int64  
 1   dsport                35 non-null     int64  
 2   proto                 35 non-null     str    
 3   state                 35 non-null     str    
 4   dur                   35 non-null     float64
 5   sbytes                35 non-null     float64
 6   dbytes                35 non-null     float64
 7   Spkts                 35 non-null     float64
 8   Dpkts                 35 non-null     float64
 9   service               35 non-null     str    
 10  Stime                 35 non-null     float64
 11  Sload                 35 non-null     float64
 12  Dload                 35 non-null     float64
 13  smeansz               35 non-null     float64
 14  dmeansz               35 non-null     float64
 15  Ltime                 35 non-null   

# Modelling

## Simulate moving data

In [21]:
from pathlib import Path
import pandas as pd

df_base = pd.read_csv(Path('data') / 'processed' / 'base_train.csv')

In [22]:
df_base.info()

<class 'pandas.DataFrame'>
RangeIndex: 1270023 entries, 0 to 1270022
Data columns (total 31 columns):
 #   Column                Non-Null Count    Dtype  
---  ------                --------------    -----  
 0   sport                 1270023 non-null  int64  
 1   dsport                1270023 non-null  int64  
 2   proto                 1270023 non-null  str    
 3   state                 1270023 non-null  str    
 4   dur                   1270023 non-null  float64
 5   sbytes                1270023 non-null  int64  
 6   dbytes                1270023 non-null  int64  
 7   service               1270023 non-null  str    
 8   Sload                 1270023 non-null  float64
 9   Dload                 1270023 non-null  float64
 10  Spkts                 1270023 non-null  int64  
 11  Dpkts                 1270023 non-null  int64  
 12  smeansz               1270023 non-null  int64  
 13  dmeansz               1270023 non-null  int64  
 14  Stime                 1270023 non-null  int64

In [23]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(df_base.drop(columns=['Label']), df_base['Label'], test_size=0.2)

In [26]:
X_train_encoded = X_train.copy()
X_test_encoded = X_test.copy()

frequency_proto = X_train['proto'].value_counts(normalize=True).to_dict()
frequency_state = X_train['state'].value_counts(normalize=True).to_dict()
frequency_service = X_train['service'].value_counts(normalize=True).to_dict()

X_train_encoded['proto_Freq'] = X_train['proto'].map(frequency_proto)
X_test_encoded['proto_Freq'] = X_test['proto'].map(frequency_proto).fillna(0)

X_train_encoded['state_Freq'] = X_train['state'].map(frequency_proto)
X_test_encoded['state_Freq'] = X_test['state'].map(frequency_proto).fillna(0)

X_train_encoded['service_Freq'] = X_train['service'].map(frequency_proto)
X_test_encoded['service_Freq'] = X_test['service'].map(frequency_proto).fillna(0)

X_train_encoded = X_train_encoded.drop(columns=['proto', 'state', 'service', "Stime", "Ltime"])
X_test_encoded = X_test_encoded.drop(columns=['proto', 'state', 'service', "Stime", "Ltime"])

In [27]:
import xgboost as xgb
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score, roc_auc_score

xgb_model = xgb.XGBClassifier(
    tree_method="hist", 
    enable_categorical=True,
    n_estimators=100,
    random_state=42,
    
)

xgb_model.fit(X_train_encoded, y_train)

,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",None
,feature_types feature_types: typing.Optional[typing.Sequence[str]].. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [28]:
y_pred = xgb_model.predict(X_test_encoded)
print('f1: ', f1_score(y_test, y_pred))
print('accuracy: ', accuracy_score(y_test, y_pred))
print('precision: ', precision_score(y_test, y_pred))
print('recall: ', recall_score(y_test, y_pred))
print('ROC-AUC: ', roc_auc_score(y_test, y_pred))

f1:  0.9687440982058546
accuracy:  0.9973937520914943
precision:  0.9704852899441869
recall:  0.9670091431803186
ROC-AUC:  0.9828636407613865


In [29]:
from sklearn.metrics import confusion_matrix

confusion_matrix(y_test, y_pred)

array([[243084,    312],
       [   350,  10259]])